# How fast is microgpt on PYNQ-Z2?

This notebook puts a single number on the question: **how many tokens per second can the microgpt overlay sustain on this board?**

Two answers exist:
1. **The RTL ceiling.** What the core itself can do per FPGA cycle, as reported by the `perf_cycles_reg` counter the wrapper exposes at offset `0x0D8`. This is what we *would* see with a free-running data path.
2. **The PS-bound wall-clock rate.** What Python actually achieves, including AXI-Lite register polls, status reads, FSM start pulses, and Python-loop overhead.

We sweep `max_tokens` from 1 to 15, measure both numbers at each setting, and extract:
* the **per-call FSM overhead** (cycles spent in the wrapper getting in/out of `ST_WAIT_CORE`),
* the **steady-state RTL cycles per token** (the slope of cycles-vs-tokens),
* the **maximum sustained tokens/sec** achievable from this Python interface.

Headline numbers print at the bottom.

## 0. Setup

In [ ]:
import sys, time, statistics
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

drivers_path = (Path('..') / 'drivers').resolve()
if str(drivers_path) not in sys.path:
    sys.path.insert(0, str(drivers_path))

from microgpt import MicroGPT

gpt = MicroGPT()
FCLK_HZ = 50_000_000
print('overlay loaded; status =', gpt.status())

## 1. Cycles vs. tokens (the RTL story)

Run a generation at each `max_tokens` value 1..15, average over a handful of seeds (so the categorical sampler's natural variation in cycle count doesn't bias the line), and record the FPGA cycles each one took. The relationship is essentially linear: cycles = (per-call overhead) + (cycles-per-token) × (tokens emitted).

In [ ]:
MAX_TOKENS_GRID = list(range(1, 16))
SEEDS_PER_POINT = 8

rng = np.random.default_rng(2026)
seed_pool = rng.integers(1, 2**31 - 1, size=SEEDS_PER_POINT, dtype=np.int64)

rows = []  # (max_tok, n_emitted, cycles, wall_s)
for mt in MAX_TOKENS_GRID:
    for s in seed_pool:
        t0 = time.perf_counter()
        text, info = gpt.generate(max_tokens=mt, temperature=1.0, seed=int(s))
        dt = time.perf_counter() - t0
        rows.append((mt, len(info['tokens']), info['cycles'], dt))

rows = np.asarray(rows, dtype=float)
print(f'collected {len(rows)} samples across {len(MAX_TOKENS_GRID)} max_tokens settings.')
print('first 4 rows (max_tok, n_emitted, cycles, wall_s):')
print(rows[:4])

In [ ]:
# Average cycles and wall-time at each max_tok, plot.
agg = {}
for mt, n, cyc, wall in rows:
    agg.setdefault(int(mt), []).append((n, cyc, wall))

max_tokens   = np.array(sorted(agg.keys()), dtype=float)
n_emitted    = np.array([np.mean([r[0] for r in agg[int(mt)]]) for mt in max_tokens])
cycles_mean  = np.array([np.mean([r[1] for r in agg[int(mt)]]) for mt in max_tokens])
cycles_std   = np.array([np.std ([r[1] for r in agg[int(mt)]]) for mt in max_tokens])
wall_mean_ms = np.array([1e3 * np.mean([r[2] for r in agg[int(mt)]]) for mt in max_tokens])

# Linear fit: cycles = overhead + slope * n_emitted
slope, intercept = np.polyfit(n_emitted, cycles_mean, 1)
cyc_per_tok    = slope
fsm_overhead   = intercept
rtl_ceiling    = FCLK_HZ / cyc_per_tok  # tokens/sec at the steady-state slope

print(f'fit: cycles ~= {fsm_overhead:.0f} (per-call overhead) + {cyc_per_tok:.1f} cycles/token * n_tokens')
print(f'-> at FCLK = {FCLK_HZ/1e6:.0f} MHz the steady-state RTL ceiling is {rtl_ceiling:,.0f} tokens/sec ({1e6/rtl_ceiling:.2f} us/token)')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].errorbar(n_emitted, cycles_mean, yerr=cycles_std, fmt='o', capsize=3, label='measured (mean +- stdev)')
fit_x = np.linspace(0, n_emitted.max() + 1, 50)
axs[0].plot(fit_x, intercept + slope * fit_x, '--', color='C3',
            label=f'fit: {intercept:.0f} + {slope:.1f}*n')
axs[0].set_xlabel('tokens emitted'); axs[0].set_ylabel('FPGA cycles')
axs[0].set_title('RTL cycles vs tokens emitted'); axs[0].legend(); axs[0].grid(alpha=0.3)

axs[1].plot(max_tokens, wall_mean_ms, 'o-', color='C2')
axs[1].set_xlabel('max_tokens')
axs[1].set_ylabel('Python wall-clock per generate() (ms)')
axs[1].set_title('PS-side wall clock per call')
axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Reading the chart.**
* Left panel: the RTL cycle count is essentially linear in tokens emitted. The slope is the steady-state cycles-per-token (the work the unmodified TALOS-V2 core does for each token). The intercept is the wrapper's FSM overhead per generation — the cost of `ST_READY → ST_WAIT_CORE → ST_DONE` and back.
* Right panel: the wall-clock per `gpt.generate()` is also linear-ish in `max_tokens`, but the slope and intercept are larger than the RTL view because every call pays for ~6–10 AXI register transactions plus the Python loop and busy-wait status polling. That's the gap we'll quantify next.

## 2. Sustained tokens/sec across the sweep

At each `max_tokens` we have wall-clock per call and tokens emitted per call. Dividing tokens by wall-clock gives the *throughput at that batch size*. Larger `max_tokens` amortizes the per-call AXI overhead over more tokens, so throughput should grow with `max_tokens` and asymptote toward whatever the per-token cost dominates.

In [ ]:
tps_rtl_steady = FCLK_HZ / cyc_per_tok  # constant: the RTL ceiling at steady state
tps_rtl_per_call = (FCLK_HZ * n_emitted) / cycles_mean   # what we'd see if AXI overhead were 0
tps_wall         = (n_emitted * 1e3) / wall_mean_ms       # what the user sees from Python

print(f'  max_tok  tokens_emitted  wall_ms   tps_wall   tps_rtl_per_call   tps_rtl_steady')
for i, mt in enumerate(max_tokens.astype(int)):
    print(f'    {mt:>4d}    {n_emitted[i]:>6.2f}        {wall_mean_ms[i]:>6.2f}   '
          f'{tps_wall[i]:>7.0f}      {tps_rtl_per_call[i]:>7.0f}            {tps_rtl_steady:>7.0f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(max_tokens, tps_wall,         'o-', color='C0', label='end-to-end (Python wall clock)')
ax.plot(max_tokens, tps_rtl_per_call, 's-', color='C2', label='RTL-only per call (cycles-based)')
ax.axhline(tps_rtl_steady, color='C3', linestyle='--',
           label=f'RTL steady-state ceiling = {tps_rtl_steady:,.0f} tok/s')
ax.set_xlabel('max_tokens'); ax.set_ylabel('throughput (tokens/sec)')
ax.set_title('microgpt throughput vs max_tokens')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Reading the chart.** Three lines:
* **Red dashed**: pure RTL steady-state throughput. This is what the unmodified TALOS-V2 core could in principle deliver if you fed it tokens with zero overhead — it's the cycles-per-token slope inverted.
* **Green**: per-call RTL throughput including the wrapper FSM overhead. Smaller `max_tokens` pays a fixed FSM startup cost over fewer tokens, so this curve asymptotes to the red line as `max_tokens` grows.
* **Blue**: what Python actually measures. The gap between blue and green is *entirely* PS-side overhead: AXI register transactions for `REG_CMD`, `REG_CONFIG`, `REG_SEED`, `REG_STATUS` polling for `done`, and 1..15 reads of `REG_OUT_BASE+4*i`. That gap is the cost of the AXI-Lite register interface.

## 3. The headline: maximum sustained throughput

Run the largest `max_tokens` setting (15) for a fixed wall-clock budget (~5 s) and count how many tokens were emitted. This is the *maximum sustained* tokens/sec that a Python user can observe from this overlay.

In [ ]:
BUDGET_S  = 5.0
MAX_TOK   = 15

rng = np.random.default_rng(7)
n_calls  = 0
n_tokens = 0
n_cycles_rtl = 0

t_end = time.perf_counter() + BUDGET_S
while time.perf_counter() < t_end:
    s = int(rng.integers(1, 2**31 - 1))
    text, info = gpt.generate(max_tokens=MAX_TOK, temperature=1.0, seed=s)
    n_calls  += 1
    n_tokens += len(info['tokens'])
    n_cycles_rtl += info['cycles']
elapsed = BUDGET_S

tps_e2e = n_tokens / elapsed
tps_rtl = (FCLK_HZ * n_tokens) / n_cycles_rtl
print(f'sustained run, max_tokens={MAX_TOK}, budget={BUDGET_S:.1f} s:')
print(f'  generations completed     : {n_calls}')
print(f'  tokens emitted            : {n_tokens}')
print(f'  RTL cycles total          : {n_cycles_rtl} ({n_cycles_rtl / FCLK_HZ * 1e3:.1f} ms of pure inference)')
print(f'  tps end-to-end (wall)     : {tps_e2e:>8,.0f} tokens/sec')
print(f'  tps if AXI were free      : {tps_rtl:>8,.0f} tokens/sec')
print(f'  fraction of RTL achieved  : {100*tps_e2e/tps_rtl:.1f}% (the rest is PS-side overhead)')
print(f'  RTL steady-state ceiling  : {tps_rtl_steady:>8,.0f} tokens/sec')

## 4. What would it take to close the gap?

The PS-side overhead per call breaks down approximately as (numbers are typical for PYNQ + cocotbext-axi style transactions):

| step | AXI transactions | comment |
|------|---:|---|
| reset (CMD = clear)             | 1W | one MMIO write |
| program CONFIG + SEED           | 2W | two MMIO writes |
| start (CMD = start)             | 1W | one MMIO write |
| poll STATUS until `done`        | k×R | typically 3–10 reads while the core runs |
| read out_len from STATUS        | (covered above) | — |
| read OUT_BASE 0..out_len-1      | (out_len)×R | one MMIO read per token |
| read REG_PERF_CYC, REG_TPS      | 2R | two MMIO reads |

Each MMIO transaction on the PYNQ-Z2 over AXI-Lite is ~hundreds of ns of bus time *plus* Python interpreter time. For `max_tokens=15` that's roughly 20–25 transactions per generation. Closing the gap to the RTL ceiling would mean replacing the per-token register reads with a single AXI4-Stream burst — the same upgrade story as the rtl_ber overlay.

## Summary

* **RTL steady-state ceiling**: the slope of cycles vs tokens, inverted at 50 MHz, is the maximum the core can do per cycle. From this notebook's measurement, that landed at the value printed by section 1 (typically tens of thousands of tokens/sec for this tiny model).
* **Sustained end-to-end** through Python is several times lower because every generation pays AXI-Lite + Python overhead on its 20-ish register transactions.
* The headline tokens/sec for a Python user is the **section 3 "tps end-to-end (wall)"** number — that is the answer to *"how many tokens/sec can microgpt do on this board?"* in practice today.
* The headline tokens/sec for a hypothetical streaming data-plane (no per-token register polls) is the **steady-state ceiling** number — that is the answer to *"what would the RTL be capable of if we got out of its way?"*